In [1]:
# 1. imports (including the shared module)
import sys
sys.path.append('..')
from src.preprocessing.cv_pipeline import preprocess_fold, REDUCED_FEATURES

import pandas as pd
import numpy as np
import shap
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# 2. reload the (now-corrected) processed data
X_ext_train = pd.read_csv('../data/processed/X_ext_train.csv', index_col=0)
y_train = pd.read_csv('../data/processed/y_train.csv', index_col=0).squeeze()

X_ext_train.describe()

c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Age,Gender,ESR,CRP,RF,Anti-CCP,HLA-B27,ANA,Anti-Ro,Anti-La,Anti-dsDNA,Anti-Sm,C3,C4,RF_was_missing,Anti-CCP_was_missing,Inflammation_Score,C3_C4_Ratio,Autoantibody_Count
count,9.668000e+03,9668.000000,9.668000e+03,9.668000e+03,9.668000e+03,9.668000e+03,9668.000000,9668.000000,9668.000000,9668.000000,9668.000000,9668.000000,9.668000e+03,9.668000e+03,9668.000000,9668.000000,9.668000e+03,9.668000e+03,9.668000e+03
mean,-5.806048e-17,0.512722,-5.144600e-17,1.293499e-16,1.807959e-16,-3.729835e-16,0.613619,0.628544,0.575924,0.588238,0.551940,0.554096,-3.878661e-16,-1.638923e-16,0.110364,0.267998,2.792783e-17,1.403741e-16,3.645316e-16
std,1.000052e+00,0.499864,1.000052e+00,1.000052e+00,1.000052e+00,1.000052e+00,0.447271,0.413501,0.435301,0.431291,0.397798,0.384357,1.000052e+00,1.000052e+00,0.313359,0.442939,1.000052e+00,1.000052e+00,1.000052e+00
min,-1.688519e+00,0.000000,-1.707191e+00,-1.327016e+00,-1.799537e+00,-1.914470e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-2.377597e+00,-1.782840e+00,0.000000,0.000000,-1.592166e+00,-1.102925e+00,-3.186710e+00
25%,-8.979895e-01,0.000000,-1.002361e+00,-1.118022e+00,-8.292035e-01,-7.367474e-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-7.179719e-01,-7.614049e-01,0.000000,0.000000,-1.074945e+00,-6.479672e-01,-7.048681e-01
50%,5.472590e-03,1.000000,2.663335e-01,2.354631e-01,-3.852380e-02,-2.702572e-02,1.000000,0.807538,0.661116,0.683380,0.558158,0.558674,3.904989e-02,-2.878507e-03,0.000000,0.000000,3.410547e-01,-3.646195e-01,3.314607e-02
75%,8.524683e-01,1.000000,8.301977e-01,8.425411e-01,8.180207e-01,7.237107e-01,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,7.378393e-01,7.683056e-01,0.000000,1.000000,8.455574e-01,2.747467e-01,7.479667e-01
max,1.699464e+00,1.000000,1.746477e+00,1.728194e+00,1.850232e+00,1.977168e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.135418e+00,1.926583e+00,1.000000,1.000000,1.786695e+00,5.324724e+00,2.256994e+00


In [2]:
%pip install xgboost
%pip install shap

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import shap
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

X_ext_train = pd.read_csv('../data/processed/X_ext_train.csv', index_col=0)
y_train = pd.read_csv('../data/processed/y_train.csv', index_col=0).squeeze()

rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_ext_train, y_train)

explainer_rf = shap.TreeExplainer(rf)
shap_values_rf = explainer_rf.shap_values(X_ext_train)

print(type(shap_values_rf))
print(len(shap_values_rf) if isinstance(shap_values_rf, list) else shap_values_rf.shape)

<class 'numpy.ndarray'>
(9668, 19, 7)


In [4]:
import numpy as np

# shap_values_rf shape: (9668 samples, 19 features, 7 classes)
# mean(|SHAP|) across samples AND classes -> one importance score per feature
mean_abs_shap_rf = np.abs(shap_values_rf).mean(axis=(0, 2))

shap_importance_rf = pd.Series(mean_abs_shap_rf, index=X_ext_train.columns).sort_values(ascending=False)
shap_importance_rf

ESR                     0.068621
Inflammation_Score      0.050435
RF                      0.042925
Anti-CCP                0.035878
CRP                     0.032537
HLA-B27                 0.025768
C4                      0.018467
C3                      0.017858
ANA                     0.013869
Anti-Ro                 0.011772
Anti-La                 0.010646
Autoantibody_Count      0.009510
C3_C4_Ratio             0.007544
Anti-Sm                 0.006651
Anti-dsDNA              0.006212
Age                     0.002660
Anti-CCP_was_missing    0.001539
RF_was_missing          0.001073
Gender                  0.000644
dtype: float64

In [5]:
xgb = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                     objective='multi:softprob', eval_metric='mlogloss',
                     random_state=42, n_jobs=-1)
xgb.fit(X_ext_train, y_train)

explainer_xgb = shap.TreeExplainer(xgb)
shap_values_xgb = explainer_xgb.shap_values(X_ext_train, check_additivity=False)

print(type(shap_values_xgb))
print(shap_values_xgb.shape if hasattr(shap_values_xgb, 'shape') else len(shap_values_xgb))

<class 'numpy.ndarray'>
(9668, 19, 7)


In [6]:
mean_abs_shap_xgb = np.abs(shap_values_xgb).mean(axis=(0, 2))
shap_importance_xgb = pd.Series(mean_abs_shap_xgb, index=X_ext_train.columns).sort_values(ascending=False)
shap_importance_xgb

ESR                     1.187907
RF                      0.847746
CRP                     0.792252
Anti-CCP                0.774266
Inflammation_Score      0.644995
C3                      0.507422
HLA-B27                 0.425372
C4                      0.293883
Anti-Ro                 0.266825
ANA                     0.265005
Anti-La                 0.254835
C3_C4_Ratio             0.225637
Anti-Sm                 0.173664
Autoantibody_Count      0.157964
Age                     0.148880
Anti-dsDNA              0.141578
Anti-CCP_was_missing    0.091189
RF_was_missing          0.078647
Gender                  0.042207
dtype: float32

In [7]:
comparison = pd.DataFrame({
    'RF_importance': shap_importance_rf,
    'RF_rank': shap_importance_rf.rank(ascending=False).astype(int),
    'XGB_importance': shap_importance_xgb,
    'XGB_rank': shap_importance_xgb.rank(ascending=False).astype(int),
}).sort_values('RF_rank')

comparison

,RF_importance,RF_rank,XGB_importance,XGB_rank
ESR,0.068621,1,1.187907,1
Inflammation_Score,0.050435,2,0.644995,5
RF,0.042925,3,0.847746,2
Anti-CCP,0.035878,4,0.774266,4
CRP,0.032537,5,0.792252,3
HLA-B27,0.025768,6,0.425372,7
C4,0.018467,7,0.293883,8
C3,0.017858,8,0.507422,6
ANA,0.013869,9,0.265005,10
Anti-Ro,0.011772,10,0.266825,9


**SHAP feature comparison (RF vs XGBoost):** Both models independently agree on the top 8 features (ESR, RF, Anti-CCP, CRP, HLA-B27, C3, C4, Inflammation_Score) and on Anti-Ro/Anti-La as moderately important (ranks 10-11 for both). We retain these 11 features — those ranked in the top ~60% by both models — for the reduced feature set, dropping C3_C4_Ratio, Autoantibody_Count, Anti-Sm, Age, Anti-dsDNA, both missingness flags, and Gender, all of which ranked in the bottom third for both models. Note: Anti-dsDNA and Anti-Sm are dropped despite being clinically SLE-specific, likely because their signal is already captured by ANA/C3/C4 — worth monitoring SLE recall in the reduced-feature retrain.

In [8]:
reduced_features = ['ESR', 'RF', 'Anti-CCP', 'CRP', 'HLA-B27', 'C3', 'C4',
                     'Inflammation_Score', 'ANA', 'Anti-Ro', 'Anti-La']

X_reduced_train = X_ext_train[reduced_features]
X_reduced_test = pd.read_csv('../data/processed/X_ext_test.csv', index_col=0)[reduced_features]

X_reduced_train.shape, X_reduced_test.shape

((9668, 11), (2417, 11))

In [9]:
import sys
sys.path.append('..')
from src.preprocessing.cv_pipeline import preprocess_fold, REDUCED_FEATURES

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import sys
sys.path.append('..')
from src.preprocessing.cv_pipeline import preprocess_fold, REDUCED_FEATURES

df = pd.read_excel("../data/raw/dataset.xlsx")

binary_cols = ['HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm']
for col in binary_cols:
    df[col] = df[col].map({'Positive': 1, 'Negative': 0})
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['RF_was_missing'] = df['RF'].isna().astype(int)
df['Anti-CCP_was_missing'] = df['Anti-CCP'].isna().astype(int)

train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df['Disease'], random_state=42
)
df_train_full = df.loc[train_idx].copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []
for fold_num, (tr_idx, va_idx) in enumerate(skf.split(df_train_full, df_train_full['Disease']), start=1):
    fold_train_df = df_train_full.iloc[tr_idx]
    fold_val_df = df_train_full.iloc[va_idx]

    _, _, X_ext_tr, X_ext_va, y_tr, y_va = preprocess_fold(fold_train_df, fold_val_df)

    # slice down to the SHAP-reduced feature set -- already scaled correctly, no re-fit needed
    X_red_tr = X_ext_tr[REDUCED_FEATURES]
    X_red_va = X_ext_va[REDUCED_FEATURES]

    models = {
        'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
        'XGBoost': XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                                  objective='multi:softprob', eval_metric='mlogloss',
                                  random_state=42, n_jobs=-1)
    }

    for name, model in models.items():
        model.fit(X_red_tr, y_tr)
        y_pred = model.predict(X_red_va)

        results.append({
            'fold': fold_num, 'model': name, 'features': 'reduced_shap',
            'macro_f1': f1_score(y_va, y_pred, average='macro'),
            'weighted_f1': f1_score(y_va, y_pred, average='weighted'),
            'accuracy': accuracy_score(y_va, y_pred),
            'balanced_accuracy': balanced_accuracy_score(y_va, y_pred),
        })

    print(f"Fold {fold_num} done")

cv_results_reduced = pd.DataFrame(results)
cv_results_reduced

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done


,fold,model,features,macro_f1,weighted_f1,accuracy,balanced_accuracy
0,1,Random Forest,reduced_shap,0.815297,0.819638,0.822647,0.820634
1,1,XGBoost,reduced_shap,0.812413,0.819392,0.822130,0.803393
2,2,Random Forest,reduced_shap,0.810418,0.823663,0.824716,0.809540
3,2,XGBoost,reduced_shap,0.818380,0.836029,0.837642,0.809085
4,3,Random Forest,reduced_shap,0.820467,0.832109,0.834023,0.834815
5,3,XGBoost,reduced_shap,0.821678,0.829161,0.831437,0.817959
6,4,Random Forest,reduced_shap,0.813221,0.825489,0.829281,0.829507
7,4,XGBoost,reduced_shap,0.806174,0.815677,0.820486,0.807801
8,5,Random Forest,reduced_shap,0.817431,0.826590,0.826177,0.826392
9,5,XGBoost,reduced_shap,0.813480,0.825514,0.826177,0.803217


In [11]:
summary_reduced = cv_results_reduced.groupby(['model', 'features']).agg(
    macro_f1_mean=('macro_f1', 'mean'),
    macro_f1_std=('macro_f1', 'std'),
).round(4).sort_values('macro_f1_mean', ascending=False)

summary_reduced

,,macro_f1_mean,macro_f1_std
model,features,,
Random Forest,reduced_shap,0.8154,0.0039
XGBoost,reduced_shap,0.8144,0.0059


In [12]:
y_test = pd.read_csv('../data/processed/y_test.csv', index_col=0).squeeze()

In [13]:
from sklearn.metrics import classification_report

# quick full-fit comparison to inspect per-class impact (not CV, just a diagnostic look)
disease_names = ['Ankylosing Spondylitis', 'Normal', 'Psoriatic Arthritis',
                  'Reactive Arthritis', 'Rheumatoid Arthritis',
                  "Sjögren's Syndrome", 'Systemic Lupus Erythematosus']

rf_reduced = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
rf_reduced.fit(X_reduced_train, y_train)
y_pred_reduced = rf_reduced.predict(X_reduced_test)

print(classification_report(y_test, y_pred_reduced, target_names=disease_names, digits=3))

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.738     0.616     0.672       425
                      Normal      0.868     0.863     0.866       321
         Psoriatic Arthritis      0.852     0.905     0.878       357
          Reactive Arthritis      0.675     0.786     0.726       103
        Rheumatoid Arthritis      0.832     0.875     0.853       570
          Sjögren's Syndrome      0.878     0.897     0.888       370
Systemic Lupus Erythematosus      1.000     0.982     0.991       271

                    accuracy                          0.844      2417
                   macro avg      0.835     0.846     0.839      2417
                weighted avg      0.842     0.844     0.842      2417



shaambhavi: for stacking i am going ahead with the 11 features eevn though there is a slight decrease in macro f1 because we are decreasing feature set by 42% which will make it easier to compute othr things like hyperparams tuning and all.

In [14]:
# confirm what's actually in memory right now
print(X_reduced_train.shape)
print(X_reduced_train[['HLA-B27', 'ANA']].describe())  # should show continuous values, NOT clean 0/1

(9668, 11)
           HLA-B27          ANA
count  9668.000000  9668.000000
mean      0.613619     0.628544
std       0.447271     0.413501
min       0.000000     0.000000
25%       0.000000     0.000000
50%       1.000000     0.807538
75%       1.000000     1.000000
max       1.000000     1.000000


In [15]:
for col in ['HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La']:
    n_fractional = ((X_reduced_train[col] != 0) & (X_reduced_train[col] != 1)).sum()
    print(f"{col}: {n_fractional} fractional values out of {len(X_reduced_train)}")

HLA-B27: 1555 fractional values out of 9668
ANA: 2995 fractional values out of 9668
Anti-Ro: 2317 fractional values out of 9668
Anti-La: 2390 fractional values out of 9668


In [16]:
import numpy as np

mean_abs_shap_rf = np.abs(shap_values_rf).mean(axis=(0, 2))
shap_importance_rf = pd.Series(mean_abs_shap_rf, index=X_ext_train.columns).sort_values(ascending=False)

mean_abs_shap_xgb = np.abs(shap_values_xgb).mean(axis=(0, 2))
shap_importance_xgb = pd.Series(mean_abs_shap_xgb, index=X_ext_train.columns).sort_values(ascending=False)

comparison = pd.DataFrame({
    'RF_importance': shap_importance_rf,
    'RF_rank': shap_importance_rf.rank(ascending=False).astype(int),
    'XGB_importance': shap_importance_xgb,
    'XGB_rank': shap_importance_xgb.rank(ascending=False).astype(int),
}).sort_values('RF_rank')

comparison

,RF_importance,RF_rank,XGB_importance,XGB_rank
ESR,0.068621,1,1.187907,1
Inflammation_Score,0.050435,2,0.644995,5
RF,0.042925,3,0.847746,2
Anti-CCP,0.035878,4,0.774266,4
CRP,0.032537,5,0.792252,3
HLA-B27,0.025768,6,0.425372,7
C4,0.018467,7,0.293883,8
C3,0.017858,8,0.507422,6
ANA,0.013869,9,0.265005,10
Anti-Ro,0.011772,10,0.266825,9


In [17]:
rf_reduced = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
rf_reduced.fit(X_reduced_train, y_train)
y_pred_reduced = rf_reduced.predict(X_reduced_test)

print(classification_report(y_test, y_pred_reduced, target_names=disease_names, digits=3))

                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.738     0.616     0.672       425
                      Normal      0.868     0.863     0.866       321
         Psoriatic Arthritis      0.852     0.905     0.878       357
          Reactive Arthritis      0.675     0.786     0.726       103
        Rheumatoid Arthritis      0.832     0.875     0.853       570
          Sjögren's Syndrome      0.878     0.897     0.888       370
Systemic Lupus Erythematosus      1.000     0.982     0.991       271

                    accuracy                          0.844      2417
                   macro avg      0.835     0.846     0.839      2417
                weighted avg      0.842     0.844     0.842      2417



In [18]:
import sys
sys.path.append('..')
from src.preprocessing.cv_pipeline import preprocess_fold, REDUCED_FEATURES

import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

df = pd.read_excel("../data/raw/dataset.xlsx")
binary_cols = ['HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm']
for col in binary_cols:
    df[col] = df[col].map({'Positive': 1, 'Negative': 0})
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['RF_was_missing'] = df['RF'].isna().astype(int)
df['Anti-CCP_was_missing'] = df['Anti-CCP'].isna().astype(int)

train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df['Disease'], random_state=42
)
df_train_full = df.loc[train_idx].copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

stage2_logreg_results = []
for fold_num, (tr_idx, va_idx) in enumerate(skf.split(df_train_full, df_train_full['Disease']), start=1):
    fold_train_df = df_train_full.iloc[tr_idx]
    fold_val_df = df_train_full.iloc[va_idx]

    _, _, X_ext_tr, X_ext_va, y_tr, y_va = preprocess_fold(fold_train_df, fold_val_df)
    X_red_tr = X_ext_tr[REDUCED_FEATURES]
    X_red_va = X_ext_va[REDUCED_FEATURES]

    lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    lr.fit(X_red_tr, y_tr)
    y_pred = lr.predict(X_red_va)

    stage2_logreg_results.append({
        'fold': fold_num,
        'macro_f1': f1_score(y_va, y_pred, average='macro'),
        'weighted_f1': f1_score(y_va, y_pred, average='weighted'),
    })
    print(f"Fold {fold_num} done")

logreg_df = pd.DataFrame(stage2_logreg_results)
print(logreg_df)
print("\nMean macro F1:", logreg_df['macro_f1'].mean().round(4))
print("Mean weighted F1:", logreg_df['weighted_f1'].mean().round(4))

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done
   fold  macro_f1  weighted_f1
0     1  0.773920     0.779756
1     2  0.784589     0.797531
2     3  0.789092     0.802401
3     4  0.789992     0.804402
4     5  0.799791     0.811532

Mean macro F1: 0.7875
Mean weighted F1: 0.7991


In [19]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

stage2_rf_xgb_results = []
for fold_num, (tr_idx, va_idx) in enumerate(skf.split(df_train_full, df_train_full['Disease']), start=1):
    fold_train_df = df_train_full.iloc[tr_idx]
    fold_val_df = df_train_full.iloc[va_idx]

    _, _, X_ext_tr, X_ext_va, y_tr, y_va = preprocess_fold(fold_train_df, fold_val_df)
    X_red_tr = X_ext_tr[REDUCED_FEATURES]
    X_red_va = X_ext_va[REDUCED_FEATURES]

    models = {
        'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=1),
        'XGBoost': XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                                  objective='multi:softprob', eval_metric='mlogloss',
                                  random_state=42, n_jobs=1)
    }

    for name, model in models.items():
        model.fit(X_red_tr, y_tr)
        y_pred = model.predict(X_red_va)
        stage2_rf_xgb_results.append({
            'fold': fold_num, 'model': name,
            'macro_f1': f1_score(y_va, y_pred, average='macro'),
            'weighted_f1': f1_score(y_va, y_pred, average='weighted'),
        })
    print(f"Fold {fold_num} done")

pd.DataFrame(stage2_rf_xgb_results).groupby('model')[['macro_f1', 'weighted_f1']].mean().round(4)

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done


,macro_f1,weighted_f1
model,,
Random Forest,0.8154,0.8255
XGBoost,0.8144,0.8252
